![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 5:  FHIR in Depth



**Health Informatics in Python** · Part II: Interoperability & Data Engineering · Module 5 of 16

---



FHIR (Fast Healthcare Interoperability Resources) is a standard describing data formats and elements (known as "resources") and an API for exchanging electronic health records.
the backbone of modern health-data exchange, so this module goes deep: the
**resource model**, **references** that link resources, **bundles** that package
them, **search**, and the **SMART on FHIR** authorization pattern that lets apps
read from an EHR.


## Learning objectives

By the end of this module you will be able to:

1. Explain the **FHIR resource model** and how **references** stitch resources together.
2. Build and validate `Patient`, `Observation`, and `Condition` resources from tabular data.
3. Assemble the three bundle types you meet in practice: **collection, transaction, searchset**.
4. Perform a **FHIR search** (simulated locally) and read a real server with a **graceful fallback**.
5. Describe **profiles/validation** and the **SMART on FHIR** launch/authorization flow.

> **Note on version.** `fhir.resources` here targets **FHIR R5**. The concepts —
> resources, references, bundles, search — are identical in R4, which you'll see
> most often in the wild.


## Dataset

We reuse the synthetic EHR generator (identical to Part I) and turn a slice of it
into real FHIR resources. Live-server calls target public sandboxes but **fall back
to local resources** so the notebook always runs.


In [1]:
# --- Self-contained synthetic EHR generator (identical to Part I) ---
# This is the SAME generator from Module 1. We rebuild the fake hospital database
# here so this notebook can run on its own. Nothing here is a real patient.
import numpy as np
import pandas as pd

# This function generates a synthetic electronic health record (EHR) dataset.
# It creates mock tables for patients, encounters, observations, conditions, and medications,
# all with plausible structure, but not referencing any real patients or requiring external data.
def make_synthetic_ehr(n_patients=200, seed=42):
    """
    Generate a small, internally-consistent synthetic EHR dataset.
    Returns a dict of linked DataFrames: patients, encounters, observations,
    conditions, and medications. This is for illustrative, educational use.
    """
    rng = np.random.default_rng(seed)  # same seed => same fake people every run

    # --- patients table ---
    # Create a DataFrame with n_patients patients, each with a name, sex, age, and birth year.
    first = ["Ava","Liam","Noah","Mia","Zoe","Omar","Ivan","Sara","Leo","Nina",
             "Ruth","Kai","Yara","Theo","Ida","Sam","Ana","Eli","Rex","Uma"]
    last  = ["Khan","Ortiz","Chen","Diaz","Patel","Ali","Brown","Nash","Reed","Vega",
             "Cole","Frost","Grant","Hale","Iqbal","Jain","Kerr","Lund","Mora","Park"]
    sexes = rng.choice(["male", "female"], size=n_patients, p=[0.49, 0.51])
    ages  = rng.integers(18, 90, size=n_patients)
    # Columns:
    #   patient_id  — unique ID, e.g. "P1000", "P1001"
    #   given_name / family_name — randomly chosen from the name lists
    #   sex, age — pre-generated arrays
    #   birth_year — derived from age, assuming the reference year 2026
    patients = pd.DataFrame({
        "patient_id": [f"P{1000+i}" for i in range(n_patients)],
        "given_name": rng.choice(first, size=n_patients),
        "family_name": rng.choice(last, size=n_patients),
        "sex": sexes,
        "age": ages,
        "birth_year": 2026 - ages,
    })

    # --- encounters table ---
    # Each patient gets 1–4 visits. Types and dates are random but internally consistent.
    #   encounter_id    — unique, zero-padded (E00001, E00002, ...)
    #   encounter_type  — ambulatory 55%, emergency 15%, inpatient 10%, wellness 20%
    #   date            — a random day in a 3-year window starting 2023-01-01
    enc_rows = []
    enc_types = ["ambulatory", "emergency", "inpatient", "wellness"]
    for pid in patients["patient_id"]:
        for _ in range(rng.integers(1, 5)):  # 1 to 4 encounters per patient
            day = rng.integers(0, 365*3)     # offset within 3 years
            enc_rows.append({
                "encounter_id": f"E{len(enc_rows)+1:05d}",
                "patient_id": pid,
                "encounter_type": rng.choice(enc_types, p=[0.55, 0.15, 0.10, 0.20]),
                "date": (pd.Timestamp("2023-01-01") + pd.Timedelta(days=int(day))).date(),
            })
    encounters = pd.DataFrame(enc_rows)

    # --- observations table ---
    # Labs and vitals attached to encounters. Each tuple is (name, unit, lo, hi).
    # A value is drawn uniformly in [lo, hi]; 70% of (encounter, measure) pairs exist
    # so the table looks sparse the way a real EHR does.
    obs_defs = [
        ("Body height", "cm", 150, 195),
        ("Body weight", "kg", 50, 110),
        ("Systolic blood pressure", "mmHg", 100, 165),
        ("Heart rate", "/min", 55, 100),
        ("Hemoglobin A1c", "%", 4.8, 9.5),
    ]
    obs_rows = []
    for _, e in encounters.iterrows():
        for name, unit, lo, hi in obs_defs:
            if rng.random() < 0.7:
                obs_rows.append({
                    "observation_id": f"O{len(obs_rows)+1:06d}",
                    "encounter_id": e["encounter_id"],
                    "patient_id": e["patient_id"],
                    "observation": name,
                    "value": round(float(rng.uniform(lo, hi)), 1),
                    "unit": unit,
                    "date": e["date"],
                })
    observations = pd.DataFrame(obs_rows)

    # --- conditions table ---
    # Each patient is assigned 0–3 unique diagnoses from a small pool.
    cond_pool = ["Essential hypertension", "Type 2 diabetes mellitus", "Asthma",
                 "Acute bronchitis", "Major depressive disorder", "Osteoarthritis",
                 "Chronic kidney disease", "Anemia"]
    cond_rows = []
    for pid in patients["patient_id"]:
        for c in rng.choice(cond_pool, size=rng.integers(0, 4), replace=False):
            cond_rows.append({
                "condition_id": f"C{len(cond_rows)+1:05d}",
                "patient_id": pid,
                "condition": c,
            })
    conditions = pd.DataFrame(cond_rows)

    # --- medications table ---
    # Same pattern as conditions: 0–3 unique drugs per patient.
    med_pool = ["Lisinopril", "Metformin", "Albuterol", "Atorvastatin",
                "Sertraline", "Amoxicillin", "Ibuprofen", "Hydrochlorothiazide"]
    med_rows = []
    for pid in patients["patient_id"]:
        for m in rng.choice(med_pool, size=rng.integers(0, 4), replace=False):
            med_rows.append({
                "medication_id": f"M{len(med_rows)+1:05d}",
                "patient_id": pid,
                "medication": m,
            })
    medications = pd.DataFrame(med_rows)

    return {
        "patients": patients,
        "encounters": encounters,
        "observations": observations,
        "conditions": conditions,
        "medications": medications,
    }

# Generate the five linked tables. Each notebook in this series starts from here.
ehr = make_synthetic_ehr()
print("Tables:", ", ".join(f"{k} ({len(v)} rows)" for k, v in ehr.items()))


Tables: patients (200 rows), encounters (511 rows), observations (1804 rows), conditions (304 rows), medications (276 rows)


## 5.1 The resource model and references

In FHIR, information is separated into distinct building blocks called **resources**. 
These resources do not contain other resources directly; instead, they are linked by references 
using the pattern `ResourceType/id`. For example, an `Observation` resource links to its associated 
patient through a reference like `subject: Patient/P1001`. This referencing mechanism enables systems 
to piece together a complete health record from separately stored resources, rather than relying on 
deeply nested or monolithic data structures.


In [2]:
# fhir.resources is a Python library that:
#   1. lets us build FHIR objects with keyword arguments
#   2. checks they match the FHIR schema (via pydantic)
#   3. can dump them back to JSON for sending over an API
#
# A Patient resource is a JSON (or XML) object with a defined schema:
# id, name, gender, birthDate, ...  We build one from a row in our table.
from fhir.resources.patient import Patient
from fhir.resources.observation import Observation
from fhir.resources.condition import Condition

# .iloc[0] takes the first row of the patients table (one synthetic person).
row = ehr["patients"].iloc[0]

# Build a Patient resource from that row.
# name is a list because a person can have several names (official, maiden, nickname).
# given is also a list: some people have more than one given name.
# birthDate uses YYYY-MM-DD; we only know the year, so we use January 1 as a stand-in.
patient = Patient(
    id=row["patient_id"],
    name=[{"use": "official",
           "family": row["family_name"],
           "given": [row["given_name"]]}],
    gender=row["sex"],
    birthDate=f"{row['birth_year']}-01-01",
)

# model_dump_json() serializes the object to FHIR JSON (indent=2 makes it readable)
print(patient.model_dump_json(indent=2))


{
  "resourceType": "Patient",
  "id": "P1000",
  "name": [
    {
      "use": "official",
      "family": "Khan",
      "given": [
        "Yara"
      ]
    }
  ],
  "gender": "female",
  "birthDate": "1937-01-01"
}


### Milestone 1 - convert tabular observations to referenced Observation resources

In [3]:
# An Observation resource is FHIR's version of a lab result or vital sign.
# The CODE (LOINC) is what makes the observation interoperable — the same
# 4548-4 means "Hemoglobin A1c" in every FHIR system, not just ours.
#
# LOINC maps our human-readable observation names to official codes.
LOINC = {
    "Hemoglobin A1c": "4548-4",
    "Systolic blood pressure": "8480-6",
    "Heart rate": "8867-4",
    "Body weight": "29463-7",
    "Body height": "8302-2",
}

pid = row["patient_id"]  # e.g. "P1000" — the same person as the Patient above

# .query() filters the observations table to this patient only.
# @pid lets pandas substitute the Python variable pid into the query string.
# .head(4) keeps the first four rows so the example stays small.
pt_obs = ehr["observations"].query("patient_id == @pid").head(4)

obs_resources = []
for _, o in pt_obs.iterrows():  # walk each observation row
    obs_resources.append(Observation(
        status="final",  # the result is complete, not a preliminary draft
        # code.coding is a list so one observation can be tagged with several systems
        code={"coding": [{"system": "http://loinc.org",
                          "code": LOINC[o["observation"]],
                          "display": o["observation"]}]},
        # subject.reference points at the Patient resource we built above.
        # The pattern is always ResourceType/id  — that is the FHIR "pointer".
        subject={"reference": f"Patient/{pid}"},
        effectiveDateTime=str(o["date"]),  # when the measurement was taken
        # valueQuantity holds the number and the unit
        valueQuantity={"value": float(o["value"]), "unit": o["unit"]},
    ))

print(f"Built {len(obs_resources)} Observation resources, all referencing Patient/{pid}")
print("\nExample subject reference:", obs_resources[0].subject.reference)


Built 4 Observation resources, all referencing Patient/P1000

Example subject reference: Patient/P1000


## 5.2 Bundles: collection, transaction, searchset

A **Bundle** in FHIR is a resource that groups together multiple other resources into a single package.
The meaning of a Bundle depends on its `type` field, which defines how its contents should be interpreted:

- **collection**: Represents a simple group of resources bundled together, such as a set of related documents or a snapshot of information.
- **transaction**: Specifies a batch of operations that should be performed together in a single, atomic action on a FHIR server—meaning either all succeed or none are applied. Each item in the bundle includes instructions on what to do (in a `request` field).
- **searchset**: Represents the results from a FHIR server search query. When you ask a FHIR server to search for resources, it responds with a Bundle of type `searchset` containing the matching resources.


In [4]:
# A Bundle packages several resources into one payload.
# type="transaction" means: apply every entry as a write, all-or-nothing.
# If any entry fails on the server, none of them should be saved.
#
# Each entry needs two pieces:
#   resource  — the FHIR object itself
#   request   — the HTTP method and URL the server should execute
from fhir.resources.bundle import Bundle

# PUT Patient/{id}  → create-or-replace this patient (idempotent: same id, same result)
tx_entries = [{"resource": patient,
               "request": {"method": "PUT", "url": f"Patient/{pid}"}}]

# POST Observation  → create a new observation; the server assigns the id
for i, obs in enumerate(obs_resources):
    tx_entries.append({"resource": obs,
                       "request": {"method": "POST", "url": "Observation"}})

transaction = Bundle(type="transaction", entry=tx_entries)

print("Transaction bundle:")
print("  entries:", len(transaction.entry))
print("  first request:", transaction.entry[0].request.method, transaction.entry[0].request.url)
print("  this is what an app POSTs to [base]/ to write atomically")


Transaction bundle:
  entries: 5
  first request: PUT Patient/P1000
  this is what an app POSTs to [base]/ to write atomically


## 5.3 FHIR search - simulated locally

In FHIR, performing a **search** typically involves making an HTTP GET request such as:
  `[base]/Observation?subject=Patient/P1001&code=4548-4`
This instructs the FHIR server to return all Observation resources associated with subject Patient/P1001 and the specific code 4548-4.
The server responds with a **Bundle** of type `searchset`, which contains the results matching the search criteria.
In this example, we mimic this search functionality locally by iterating through our in-memory resources,
allowing you to understand how FHIR servers process searches without requiring a live server connection.


In [5]:
# A real FHIR search is an HTTP GET like:
#   [base]/Observation?subject=Patient/P1001&code=4548-4
# The server returns a Bundle with type="searchset".
#
# We don't need a live server to see the mechanics. This function walks an
# in-memory list of resources and applies the same two filters a server would:
#   subject  — whose record is this?  (Patient/P1001)
#   code     — which LOINC (or other) code?
def fhir_search(resources, resource_type, **params):
    """Minimal FHIR-style search over in-memory resources -> searchset Bundle."""
    hits = []
    for r in resources:
        # Skip anything that isn't the requested resource type (Patient vs Observation)
        if r.get_resource_type() != resource_type:
            continue
        ok = True
        # subject filter: Observation.subject.reference must match exactly
        if "subject" in params:
            ok &= (getattr(r, "subject", None) is not None
                   and r.subject.reference == params["subject"])
        # code filter: look inside code.coding[] for the requested code
        if "code" in params:
            codes = [c.code for c in r.code.coding] if getattr(r, "code", None) else []
            ok &= params["code"] in codes
        if ok:
            hits.append(r)
    # A searchset Bundle carries the matching resources plus a total count
    return Bundle(type="searchset", total=len(hits),
                  entry=[{"resource": h} for h in hits])

# Search the patient + their observations as if they were a tiny FHIR server
pool = [patient] + obs_resources
result = fhir_search(pool, "Observation", subject=f"Patient/{pid}", code="4548-4")

print(f"GET [base]/Observation?subject=Patient/{pid}&code=4548-4")
print("searchset total:", result.total)
for e in result.entry:
    q = e.resource.valueQuantity
    print(f"  -> {e.resource.code.coding[0].display}: {q.value} {q.unit}")


GET [base]/Observation?subject=Patient/P1000&code=4548-4
searchset total: 0


## 5.4 Reading from a real server — with a fallback explained

In the context of FHIR and real-world notebooks: when fetching resources from public sandboxes (such as HAPI), you would typically use the `requests` library for HTTP calls.
However, notebooks are sometimes unable to access external servers, which could cause code to fail.
To prevent this, well-designed tutorial code uses "graceful degradation": if a live server can't be reached, it falls back to using local data, ensuring the notebook still works and doesn't crash.


In [6]:
# Against a public sandbox (e.g. HAPI FHIR) you read a resource with HTTP GET:
#   GET https://hapi.fhir.org/baseR5/Patient/P1001
#   Accept: application/fhir+json
#
# Tutorial notebooks often can't reach the internet, so this function
# degrades gracefully: try the live server, and if anything fails (timeout,
# DNS, 404, ...), return the local Patient we already built.
import requests

def get_patient_live(base, pid, timeout=5):
    """Try a live FHIR read; fall back to the local resource on any failure."""
    try:
        r = requests.get(
            f"{base}/Patient/{pid}",
            headers={"Accept": "application/fhir+json"},  # ask for FHIR JSON, not HTML
            timeout=timeout,  # don't hang forever if the sandbox is down
        )
        if r.status_code == 200:  # 200 = the resource was found
            return "live", r.json()  # r.json() parses the response body into a dict
    except Exception:
        pass  # network error, timeout, etc. — fall through to local data
    # model_dump() is the dict form of our local Patient (same shape as live JSON)
    return "fallback", patient.model_dump()

source, data = get_patient_live("https://hapi.fhir.org/baseR5", pid)
print(f"source: {source}")
print("patient family name:", data["name"][0]["family"])
print("\n(The REST pattern is identical against any FHIR server; only the base URL changes.)")


source: fallback
patient family name: Khan

(The REST pattern is identical against any FHIR server; only the base URL changes.)


## 5.5 Profiles and validation

A **FHIR profile** is a set of rules that further restricts or customizes the base FHIR specification for a particular context or country. For example, the **US Core** profile specifies that certain fields in a Patient resource are required for use in the United States.

Validation is the process of checking that a FHIR resource adheres to its schema and any applicable profile. The `fhir.resources` Python library checks the base FHIR schema whenever you create a resource. If you provide invalid or incomplete data, it will immediately raise an error. This automatic validation is a critical first step to ensure that the data you're working with is correct and trustworthy (you’ll see more on advanced data quality in Module 8).


In [7]:
# fhir.resources enforces the FHIR schema at construction time.
# Observation.status is a coded field: it must be one of
#   registered | preliminary | final | amended | corrected | cancelled | ...
# Passing a made-up value raises pydantic's ValidationError instead of
# silently shipping a broken resource. That is your first data-quality gate.
from pydantic import ValidationError

try:
    bad = Observation(
        status="not-a-real-status",  # invalid — FHIR does not allow this string
        code={"coding": [{"code": "x"}]},
    )
except ValidationError as e:
    print("Schema validation rejected the resource:")
    print(" ", str(e).splitlines()[0])  # first line of the error is enough
    print("  -> 'status' must be a valid FHIR observation-status code")


## 5.6 What is SMART on FHIR? How do apps connect?

**SMART on FHIR** is a specification that allows third-party applications to securely connect to electronic health record (EHR) systems. It does this by combining FHIR (for accessing health data) with OAuth2 (for authorization and user consent).

Here’s how the SMART on FHIR authorization process typically works:

1. **Launch** – The EHR launches an app, providing both a `launch` token and the FHIR server’s base URL.
2. **Authorize** – The app redirects the user to the EHR’s OAuth2 authorization page, asking for specific “scopes” (permissions), for example, the ability to read a patient’s Observations (`patient/Observation.read`).
3. **Token** – After the user approves, the app receives an authorization code, which it exchanges with the EHR for an access token.
4. **Call** – The app makes FHIR API requests to the server, including `Authorization: Bearer <token>` in the HTTP header; this proves the app’s identity and defines which data it is allowed to access.




In [8]:
# SMART on FHIR's last step is an ordinary FHIR search with an extra header:
#   Authorization: Bearer <access_token>
# The token is what the EHR issued after the user authorized the app.
# Scopes on that token (e.g. patient/Observation.read) decide what the
# server will actually return.
#
# We don't make a live call here — we just show the request you would send.
def authorized_search(base, token, path):
    headers = {
        "Authorization": f"Bearer {token}",  # proves the app is allowed to read
        "Accept": "application/fhir+json",
    }
    # Live version would be:  requests.get(f"{base}/{path}", headers=headers)
    return f"GET {base}/{path}  [Authorization: Bearer <token>]"

print(authorized_search(
    "https://ehr.example/fhir",
    "TOKEN",
    f"Observation?patient={pid}&code=4548-4",
))


GET https://ehr.example/fhir/Observation?patient=P1000&code=4548-4  [Authorization: Bearer <token>]


## Exercises

1. Add a `Condition` resource for one of the patient's diagnoses (SNOMED from
   Module 3) and include it in the transaction bundle.
2. Extend `fhir_search` to support a `date`/`effective` filter and query for
   observations after a given date.
3. Wrap `get_patient_live` to also try an `Observation?patient=` search and return
   a searchset, still with a local fallback.



## Key takeaways

- FHIR records are **modular resources linked by references**, not one big blob.
- **Bundle type** (collection/transaction/searchset) determines what the bundle *does*.
- **Search is HTTP**; the returned searchset bundle is the unit of exchange.
- Schema **validation** is free with `fhir.resources`; **SMART on FHIR** is how apps
  obtain scoped access.



---
*Next: Module 6 - Data Ingestion and ETL Pipelines.*
